# Modelado

Se comparan varios modelos sobre la misma partición, de menor a mayor complejidad. El
orden importa: sin un baseline trivial no se sabe si un modelo complejo aporta algo.

## Partición por scaffold

Esta es la decisión metodológica más importante del proyecto. Una partición aleatoria
reparte análogos del mismo esqueleto entre entrenamiento y test, y el modelo solo
interpola. La partición por scaffold de Bemis–Murcko garantiza que los esqueletos del
test **no aparecen** en entrenamiento, que es la situación real de uso: predecir sobre
química nueva.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)
datos = Path("../datos/procesados")

X_fp = np.load(datos / "X_fp.npy")
X_desc = pd.read_parquet(datos / "X_desc.parquet")
meta = pd.read_parquet(datos / "y_meta.parquet")
y = meta["pic50"].to_numpy()

print(X_fp.shape, X_desc.shape, y.shape)

In [ ]:
def scaffold_split(scaffolds, fracs=(0.7, 0.15, 0.15), seed=RANDOM_STATE):
    """Reparte scaffolds enteros. Los grupos grandes van primero al train."""
    grupos = defaultdict(list)
    for i, s in enumerate(scaffolds):
        grupos[s].append(i)

    sets = sorted(grupos.values(), key=len, reverse=True)
    n = len(scaffolds)
    n_tr, n_va = int(fracs[0] * n), int(fracs[1] * n)

    train, val, test = [], [], []
    for g in sets:
        if len(train) + len(g) <= n_tr:
            train += g
        elif len(val) + len(g) <= n_va:
            val += g
        else:
            test += g
    return np.array(train), np.array(val), np.array(test)


idx_tr, idx_va, idx_te = scaffold_split(meta["scaffold"].tolist())

for nombre, idx in [("train", idx_tr), ("val", idx_va), ("test", idx_te)]:
    print(f"{nombre:6s} {len(idx):6d} compuestos  "
          f"{meta['scaffold'].iloc[idx].nunique():5d} scaffolds  "
          f"pIC50 medio {y[idx].mean():.2f}")

# comprobación: ningún scaffold compartido
assert not (set(meta['scaffold'].iloc[idx_tr]) & set(meta['scaffold'].iloc[idx_te]))
print("\nsin solape de scaffolds entre train y test")

```{important}
Conviene calcular también una partición **aleatoria** y reportar las dos. La diferencia
entre ambas métricas es la medida de cuánto se estaba engañando uno con la partición
aleatoria. Suele ser de 0,2–0,3 en $R^2$.
```

In [ ]:
from sklearn.preprocessing import StandardScaler

# escalado de descriptores ajustado SOLO con train
scaler = StandardScaler().fit(X_desc.iloc[idx_tr])
X_desc_s = scaler.transform(X_desc)

# matriz combinada
X = np.hstack([X_fp, X_desc_s]).astype(np.float32)

X_tr, y_tr = X[idx_tr], y[idx_tr]
X_va, y_va = X[idx_va], y[idx_va]
X_te, y_te = X[idx_te], y[idx_te]
X_tr.shape

## Modelo 0 — baseline

Predecir siempre la media. Cualquier modelo tiene que batirlo con holgura; si no,
no hay señal.

In [ ]:
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from scipy.stats import spearmanr

def evaluar(nombre, y_true, y_pred):
    return {
        "modelo": nombre,
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "MAE": mean_absolute_error(y_true, y_pred),
        "R2": r2_score(y_true, y_pred),
        "Spearman": spearmanr(y_true, y_pred).statistic,
    }

resultados = []
resultados.append(evaluar("baseline (media)", y_va, np.full_like(y_va, y_tr.mean())))
pd.DataFrame(resultados)

## Modelo 1 — regresión lineal regularizada

Ridge sobre fingerprints. Sorprendentemente competitivo en QSAR y muy rápido; sirve
como segundo suelo.

In [ ]:
from sklearn.linear_model import RidgeCV

ridge = RidgeCV(alphas=np.logspace(-2, 4, 25)).fit(X_tr, y_tr)
resultados.append(evaluar("Ridge", y_va, ridge.predict(X_va)))
print("alpha elegido:", ridge.alpha_)
pd.DataFrame(resultados)

## Modelo 2 — Random Forest

Robusto, pocos hiperparámetros sensibles, maneja bien features binarias dispersas.
Es el caballo de batalla de QSAR.

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(
    n_estimators=500,
    max_features="sqrt",
    min_samples_leaf=2,
    n_jobs=-1,
    random_state=RANDOM_STATE,
).fit(X_tr, y_tr)

resultados.append(evaluar("Random Forest", y_va, rf.predict(X_va)))
pd.DataFrame(resultados)

## Modelo 3 — Gradient boosting

XGBoost o LightGBM. Normalmente el mejor de los modelos tabulares en este tipo de
problema. Se usa `early_stopping` contra el conjunto de validación.

In [ ]:
try:
    from xgboost import XGBRegressor

    xgb = XGBRegressor(
        n_estimators=3000,
        learning_rate=0.03,
        max_depth=8,
        subsample=0.8,
        colsample_bytree=0.5,
        min_child_weight=5,
        reg_lambda=1.0,
        early_stopping_rounds=100,
        n_jobs=-1,
        random_state=RANDOM_STATE,
        tree_method="hist",
    )
    xgb.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
    print("mejor iteración:", xgb.best_iteration)
    resultados.append(evaluar("XGBoost", y_va, xgb.predict(X_va)))
except ImportError:
    print("xgboost no instalado; se omite")
    xgb = None

pd.DataFrame(resultados)

## Modelo 4 — red neuronal de grafos (opcional)

Las GNN (GCN, GIN, Attentive FP, ChemProp/D-MPNN) aprenden la representación en vez de
usar fingerprints fijos. Con menos de ~5 000 compuestos rara vez superan a un gradient
boosting bien ajustado; con más datos sí.

Implementación recomendada: **ChemProp** (D-MPNN), que trae el pipeline hecho y acepta
descriptores adicionales. Se ejecuta fuera del notebook por línea de comandos:

```powershell
chemprop train --data-path datos/procesados/train.csv `
  --task-type regression --target-columns pic50 `
  --splits-column split --epochs 50 --save-dir modelos/chemprop
```

## Búsqueda de hiperparámetros

Con validación cruzada **agrupada por scaffold** dentro del conjunto de entrenamiento,
nunca con `KFold` normal.

In [ ]:
from sklearn.model_selection import GroupKFold, RandomizedSearchCV

grupos_tr = meta["scaffold"].iloc[idx_tr].to_numpy()
cv = GroupKFold(n_splits=5)

espacio = {
    "n_estimators": [300, 500, 800],
    "max_features": ["sqrt", "log2", 0.1, 0.3],
    "min_samples_leaf": [1, 2, 4, 8],
    "max_depth": [None, 20, 40],
}

busqueda = RandomizedSearchCV(
    RandomForestRegressor(n_jobs=-1, random_state=RANDOM_STATE),
    espacio, n_iter=20, cv=cv, scoring="neg_root_mean_squared_error",
    random_state=RANDOM_STATE, n_jobs=1, verbose=1,
)
# busqueda.fit(X_tr, y_tr, groups=grupos_tr)
# print(busqueda.best_params_, -busqueda.best_score_)

## Selección del modelo final

Se elige por rendimiento en **validación**, y solo entonces se mira el test una única vez
(capítulo 7). Tocar el test para elegir modelo lo convierte en un segundo conjunto de
validación y las métricas dejan de ser estimaciones honestas del error.

In [ ]:
import joblib
from pathlib import Path

tabla = pd.DataFrame(resultados).sort_values("RMSE")
display(tabla)

mejor_nombre = tabla.iloc[0]["modelo"]
modelos = {"Ridge": ridge, "Random Forest": rf, "XGBoost": xgb}
mejor = modelos.get(mejor_nombre)

Path("../modelos").mkdir(exist_ok=True)
if mejor is not None:
    joblib.dump({"modelo": mejor, "scaler": scaler,
                 "idx_test": idx_te, "nombre": mejor_nombre},
                "../modelos/modelo_final.joblib")
    print("guardado modelo:", mejor_nombre)